# NCLE Examples: Cleaned and Incomplete 1ZMR

## Goal

Apply [Workflow 1](workflow1_native_ncle.ipynb) to the same cleaned all-atom structure with two entanglement detection methods (`ent_detection_method=3` and `2`), and to a copy with **five consecutive residues removed from an NCLE loop**.

| Example | Input in `assets/test_structures/` | Change |
|---------|-----------------------------------|--------|
| 1. Cleaned 1ZMR, method 3 | `1zmr_model_clean.pdb` | Same structure; `ent_detection_method=3` (both GLN and TLN with the same termini) |
| 2. Cleaned 1ZMR, method 2 | `1zmr_model_clean.pdb` | Same structure and settings as Workflow 1 (`ent_detection_method=2`) |
| 3. Missing loop segment | `1zmr_model_clean_missing_59-63.pdb` | All atoms of chain A residues 59–63 removed; other records, coordinates, and numbering unchanged |

Residues **59–63** lie strictly inside the cleaned structure's representative NCLE loop **55–99** (C-terminal crossing **+108**). They are also an unstructured segment (simplified DSSP: coil). This is an intentionally incomplete input, not a rebuilt or relaxed structure. NCLEdetector excludes candidate loops containing three or more consecutive missing residues, so loops spanning this gap should not survive detection. Other NCLEs can remain.

> **Environment:** Select the `ncledetector` kernel, as in Workflows 1–4. Install the notebook viewer once in that environment with `python -m pip install nglview ipywidgets`. Run all cells in order. Detection can take several minutes per structure. Feature calculation uses UniProt accession **P00558** and requires internet access. Trust the notebook and enable Jupyter widgets for the interactive outputs. The NGLView structure widget runs directly in the notebook; no separate server is needed.

Each example displays only its Representative NCLEs table, followed by an interactive structure viewer. Tables are searchable, sortable by clicking column headings, and scrollable. Each viewer has an NCLE selector. All calculation stages still run, and full pipe-separated CSVs are saved under `$OUTDIR/<example>/`.

## Setup

Inputs are bundled with the repository; no datastore download is needed. Change `OUTDIR` if desired. Tables use pandas and IPython; structures use NGLView with an ipywidgets NCLE selector.

In [ ]:
import os
from html import escape
from pathlib import Path

import pandas as pd
import nglview as nv
import ipywidgets as widgets
from IPython.display import HTML, Markdown, display

from NCLEdetector.gaussian_entanglement import GaussianEntanglement
from NCLEdetector.clustering import ClusterNativeEntanglements
from NCLEdetector.entanglement_features import FeatureGen

REPO = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'assets/test_structures/1zmr_model_clean.pdb').is_file()
)
DATASTORE = Path(os.environ.get('DATASTORE', REPO / 'DataStore'))
OUTDIR = DATASTORE / 'outputs/notebook_tmp/NCLEexamples'
STRUCTDIR = REPO / 'assets/test_structures'
CLEAN_PDB = STRUCTDIR / '1zmr_model_clean.pdb'
MISSING_PDB = STRUCTDIR / '1zmr_model_clean_missing_59-63.pdb'
CHAIN = 'A'
UNIPROT = 'P00558'
PDBID = '1ZMR'

for pdb_file in (CLEAN_PDB, MISSING_PDB):
    if not pdb_file.is_file():
        raise FileNotFoundError(pdb_file)
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f'STRUCTDIR: {STRUCTDIR}')
print(f'OUTDIR:    {OUTDIR}')

In [ ]:
def show_table(frame, title):
    display(Markdown(f'### {title} ({len(frame)} rows)'))
    document = '''<!doctype html><html><head><meta charset="utf-8">
    <style>
    body {font: 13px sans-serif; color: #222; margin: 8px;}
    input {padding: 6px; width: 280px;} #count {margin-left: 12px;}
    .scroll {overflow: auto; max-height: 300px; margin-top: 8px;}
    table {border-collapse: collapse; white-space: nowrap; font-size: 12px;}
    td, th {padding: 6px 10px; border: 1px solid #ddd; text-align: left;}
    th {position: sticky; top: 0; background: #eaf0f6;}
    th button {font: inherit; font-weight: bold; cursor: pointer; border: 0; background: none;}
    tr:nth-child(even) {background: #f7f7f7;}
    </style></head><body>
    <label>Search <input id="search" type="search" placeholder="Filter all columns"></label>
    <span id="count" aria-live="polite"></span><div class="scroll">'''
    document += frame.to_html(index=False, border=0, na_rep='', escape=True)
    document += '''</div><script>
    const table = document.querySelector('table');
    const body = table.tBodies[0];
    const rows = Array.from(body.rows);
    function filterRows() {
      const query = document.getElementById('search').value.toLowerCase();
      rows.forEach(row => row.hidden = !row.textContent.toLowerCase().includes(query));
      const visible = rows.filter(row => !row.hidden).length;
      document.getElementById('count').textContent = `${visible} / ${rows.length} rows`;
    }
    document.getElementById('search').addEventListener('input', filterRows);
    table.querySelectorAll('th').forEach((header, column) => {
      const button = document.createElement('button');
      button.textContent = header.textContent + ' (sort)';
      header.replaceChildren(button);
      let direction = 1;
      button.addEventListener('click', () => {
        rows.sort((a, b) => {
          const left = a.cells[column].textContent.trim();
          const right = b.cells[column].textContent.trim();
          const numeric = left !== '' && right !== '' &&
                          Number.isFinite(Number(left)) && Number.isFinite(Number(right));
          return direction * (numeric ? Number(left) - Number(right)
            : left.localeCompare(right, undefined, {numeric: true}));
        });
        rows.forEach(row => body.appendChild(row));
        direction *= -1;
      });
    });
    filterRows();
    </script></body></html>'''
    display(HTML(
        f'<iframe title="{escape(title, quote=True)}" srcdoc="{escape(document, quote=True)}" '
        'width="100%" height="390" style="border: 1px solid #ddd" '
        'sandbox="allow-scripts"></iframe>'
    ))

In [ ]:
def show_ncles(pdb_file, representatives, highlight=None):
    ncles = []
    for number, row in representatives.reset_index(drop=True).iterrows():
        crossings = []
        labels = []
        for column, terminus in [('crossingsN', 'N'), ('crossingsC', 'C')]:
            if pd.notna(row[column]) and str(row[column]).strip():
                tokens = str(row[column]).split(',')
                crossings.extend(abs(int(float(token.strip()))) for token in tokens)
                labels.append(f'{terminus}-cross: {row[column]}')
        start, end = int(row['i']), int(row['j'])
        ncles.append({
            'label': f'NCLE {number + 1}: loop {start}-{end} | ' + ' | '.join(labels),
            'start': start, 'end': end, 'crossings': crossings,
        })

    selected = 0 if ncles else -1
    reference_index = None
    if highlight is not None:
        start, end, crossing = highlight
        selected = next((i for i, ncle in enumerate(ncles)
                         if (ncle['start'], ncle['end']) == (start, end)
                         and abs(crossing) in ncle['crossings']), None)
        if selected is None:
            reference_index = selected = len(ncles)
            ncles.append({
                'label': f'Reference NCLE lost from detection: loop {start}-{end} | crossing {abs(crossing)}',
                'start': start, 'end': end, 'crossings': [abs(crossing)],
            })

    view = nv.show_file(str(pdb_file), default_representation=False)
    view.layout = widgets.Layout(width='100%', height='500px')
    selector = widgets.Dropdown(
        options=[('Full structure', -1)] + [(ncle['label'], i) for i, ncle in enumerate(ncles)],
        value=selected, description='NCLE:',
        layout=widgets.Layout(width='100%'),
    )

    legend = widgets.HTML()

    def update(change=None):
        legend.value = (
            'Orange VDW spheres: native contacts (i, j) | Red: loop interior | Blue VDW spheres: absolute crossing residues'
            if ncles else 'No representative NCLEs retained. Showing the input structure.'
        )
        if reference_index is not None and selector.value == reference_index:
            legend.value += '<br><b>Reference only: this NCLE was not retained for this structure. Missing residues are not rebuilt.</b>'
        view.clear_representations()
        view.add_cartoon(selection=f':{CHAIN}', color='#bfbfbf')
        if selector.value >= 0:
            ncle = ncles[selector.value]
            if ncle['end'] - ncle['start'] > 1:
                loop = f"{ncle['start'] + 1}-{ncle['end'] - 1}:{CHAIN}"
                view.add_cartoon(selection=loop, color='#E57373')
            contacts = f"{ncle['start']}:{CHAIN} or {ncle['end']}:{CHAIN}"
            view.add_spacefill(selection=contacts, color='#FB8C00',
                               radius_type='vdw', radius_scale=1.0)
            if ncle['crossings']:
                crossings = ' or '.join(f'{residue}:{CHAIN}' for residue in ncle['crossings'])
                view.add_cartoon(selection=crossings, color='#1E88E5')
                view.add_spacefill(selection=crossings, color='#1E88E5',
                                   radius_type='vdw', radius_scale=1.0)

    selector.observe(update, names='value')
    update()
    view.center(selection=f':{CHAIN}')
    viewer = widgets.VBox([selector, legend, view])
    display(viewer)
    return viewer

In [ ]:
def make_outdirs(example):
    outdir = OUTDIR / example
    for name in ('NCLE', 'HQ_NCLE', 'clustered_HQ_NCLE', 'clustered_HQ_NCLE_features'):
        (outdir / name).mkdir(parents=True, exist_ok=True)
    return outdir


def workflow1(pdb_file, outdir, ID, ge):
    # Continues from the detection CSV written by ge.calculate_native_entanglements
    raw_file = outdir / 'NCLE' / f'{ID}_NCLE.csv'
    raw = pd.read_csv(raw_file, sep='|', dtype={'crossingsN': str, 'crossingsC': str})
    detected = raw.loc[raw['ENT'] == True].reset_index(drop=True)

    ge.select_high_quality_entanglements(
        str(raw_file), str(pdb_file), outdir=str(outdir / 'HQ_NCLE'),
        ID=ID, model='EXP', chain=CHAIN
    )
    hq_file = outdir / 'HQ_NCLE' / f'{ID}.csv'
    high_quality = pd.read_csv(hq_file, sep='|', dtype={'crossingsN': str, 'crossingsC': str})

    clustering = ClusterNativeEntanglements(organism='Ecoli', cut_off=None)
    clustering.Cluster_NativeEntanglements(
        str(hq_file), ID=ID, outdir=str(outdir / 'clustered_HQ_NCLE'), chain=CHAIN
    )
    cluster_file = outdir / 'clustered_HQ_NCLE' / f'{ID}.csv'
    representatives = pd.read_csv(
        cluster_file, sep='|', dtype={'crossingsN': str, 'crossingsC': str}
    )
    show_table(representatives, 'Representative NCLEs')

    FeatureGen(logdir=str(OUTDIR / 'logs')).get_uent_features(
        pdb_file=str(pdb_file), outdir=str(outdir / 'clustered_HQ_NCLE_features'),
        cluster_file=str(cluster_file), gene=UNIPROT, chain=CHAIN, pdbid=PDBID
    )
    feature_file = outdir / 'clustered_HQ_NCLE_features' / f'{UNIPROT}_{PDBID}_{CHAIN}_uent_features.csv'
    features = pd.read_csv(feature_file, sep='|')
    print(f'CSV outputs: {outdir}')
    return {'detected': detected, 'high_quality': high_quality,
            'representatives': representatives, 'features': features}

## Example 1. Cleaned 1ZMR with detection method 3

Workflow 1 settings (all-atom contacts, `g_threshold=0.6`, `density=1.0`, experimental-structure filtering, and *E. coli* clustering) with `ent_detection_method=3`, which requires both the Gauss linking number (GLN) and the total linking number (TLN) to detect an entanglement with the same termini. The viewer initially highlights **(55, 99, 108)** when it is retained, otherwise the first representative: native-contact residues **55** and **99** in orange, loop interior **56–98** in red, and crossing residue **108** in blue.

In [ ]:
outdir = make_outdirs('cleaned_method3')
ge = GaussianEntanglement(
    g_threshold=0.6, density=1.0, Calpha=False, CG=False, ent_detection_method=3
)
ge.calculate_native_entanglements(
    pdb_file=str(CLEAN_PDB), outdir=str(outdir / 'NCLE'), ID='1ZMR_method3', chain=CHAIN
)
cleaned_m3 = workflow1(CLEAN_PDB, outdir, '1ZMR_method3', ge)

In [ ]:
cleaned_m3_viewer = show_ncles(CLEAN_PDB, cleaned_m3['representatives'], highlight=(55, 99, 108))

## Example 2. Cleaned 1ZMR with detection method 2

The same cleaned structure and settings as Example 1, except detection method `2` (the Workflow 1 default), which uses TLN alone. The viewer initially highlights **(55, 99, 108)**, using the same colors as Example 1. Outputs are kept separate from Example 1.

In [ ]:
outdir = make_outdirs('cleaned')
ge = GaussianEntanglement(
    g_threshold=0.6, density=1.0, Calpha=False, CG=False, ent_detection_method=2
)
ge.calculate_native_entanglements(
    pdb_file=str(CLEAN_PDB), outdir=str(outdir / 'NCLE'), ID='1ZMR', chain=CHAIN
)
cleaned = workflow1(CLEAN_PDB, outdir, '1ZMR', ge)

In [ ]:
cleaned_viewer = show_ncles(CLEAN_PDB, cleaned['representatives'], highlight=(55, 99, 108))

## Example 3. Five missing residues inside an NCLE loop

Run the Example 2 calculation (method `2`) on `1zmr_model_clean_missing_59-63.pdb`. Chain A jumps directly from residue **58** to **64**; no atoms were rebuilt and the remaining residues were not renumbered. In all viewers, orange VDW spheres mark the selected NCLE's native-contact residues **i** and **j**, red marks only the loop interior, and blue VDW spheres mark crossing residues using their absolute residue numbers.

The missing-residue filter acts during **detection**, before high-quality filtering and clustering. The viewer initially highlights the lost **(55, 99, 108)** NCLE as a **reference only**, using the same colors as Examples 1 and 2 and only the atoms remaining in this incomplete structure. This reference is not added to the Representative NCLEs table. A lost NCLE here reflects insufficient coordinates to support its loop, not evidence that the intact protein lacks that entanglement. Outputs are kept separate from Examples 1 and 2.

In [ ]:
outdir = make_outdirs('missing_59-63')
ge = GaussianEntanglement(
    g_threshold=0.6, density=1.0, Calpha=False, CG=False, ent_detection_method=3
)
ge.calculate_native_entanglements(
    pdb_file=str(MISSING_PDB), outdir=str(outdir / 'NCLE'), ID='1ZMR_missing_59-63', chain=CHAIN
)
missing = workflow1(MISSING_PDB, outdir, '1ZMR_missing_59-63', ge)

In [ ]:
missing_viewer = show_ncles(MISSING_PDB, missing['representatives'], highlight=(55, 99, 108))

## Comparison

Compare retained counts at each stage in the text summary below. The raw detection CSV also includes non-entangled contacts; detected counts include only `ENT=True`. Detection and clustering reuse existing CSVs on reruns, as in Workflow 1; use a new `OUTDIR` after changing inputs or parameters.

In [ ]:
comparison = pd.DataFrame([
    {'Example': label, **{stage: len(table) for stage, table in result.items()}}
    for label, result in [('Cleaned 1ZMR (method 3)', cleaned_m3), ('Cleaned 1ZMR (method 2)', cleaned),
                     ('Missing A:59-63 (method 2)', missing)]
])
print(comparison.to_string(index=False))